In [ ]:
from pathlib import Path
import json
import os

WORK_DIR = Path(r"C:\Users\Playdata\Desktop\mle-01-p2-team2\홍기표")
RAW_INPUT_PATH = WORK_DIR / "input" / "recipes_10000_cleaned.jsonl"
CLASSIFIED_INPUT_PATH = WORK_DIR / "input" / "recipes_5000_classified.jsonl"
ENV_PATH = WORK_DIR / ".env"
if not ENV_PATH.is_file():
    ENV_PATH = WORK_DIR.parent / ".env"

SOURCE_MODE = "reuse"  # "reuse": 기존 분류 파일 / "classify": LLM으로 새 분류
MAX_RECIPES = 5000     # classify 모드의 읽기 개수. 첫 확인은 10 등으로 줄이세요.
LLM_BATCH_SIZE = 100
MAX_CONCURRENCY = 10
DB_BATCH_SIZE = 200

ALLOW_LLM_CALLS = False
ALLOW_FILE_WRITE = False
ALLOW_DB_WRITE = False

print("모드:", SOURCE_MODE)
print(".env 경로:", ENV_PATH)


In [ ]:
"""분류/적재 교안의 자료 준비 함수. 파일 실행만으로 외부 호출하지 않습니다."""
import json
from pathlib import Path


def read_recipe_jsonl(path, limit=None):
    if limit is not None and (type(limit) is not int or limit < 1):
        raise ValueError("limit은 1 이상 또는 None이어야 합니다.")
    rows, seen = [], set()
    with Path(path).open("r", encoding="utf-8-sig") as file:
        for line_number, line in enumerate(file, 1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
                if not isinstance(row, dict):
                    raise ValueError("JSON 객체가 아닙니다.")
                uid = row.get("recipe_uid")
                if not isinstance(uid, str) or not uid.strip():
                    raise ValueError("recipe_uid가 비어 있습니다.")
                if uid in seen:
                    raise ValueError(f"recipe_uid 중복: {uid}")
            except (ValueError, TypeError) as error:
                raise ValueError(f"{line_number}번째 줄: {error}") from error
            seen.add(uid)
            rows.append(row)
            if limit is not None and len(rows) >= limit:
                break
    if not rows:
        raise ValueError("레시피가 없습니다.")
    return rows

def make_llm_input(recipe):
    # 재료명만 사용
    ingredient_names = [
        item["name_normalized"]
        for item in recipe.get("ingredients_clean", [])
        if item.get("name_normalized")
    ]

    # 양념도 음식 판단에 도움이 될 수 있으므로 이름만 추가
    seasoning_names = [
        item["name_normalized"]
        for item in recipe.get("seasonings_clean", [])
        if item.get("name_normalized")
    ]

    ingredient_names = list(dict.fromkeys(
        ingredient_names + seasoning_names
    ))

    # 조리 단계
    steps = recipe.get("steps", [])

    # 너무 긴 레시피는 앞 4단계 + 마지막 2단계만 사용
    if len(steps) > 6:
        selected_steps = steps[:4] + steps[-2:]
    else:
        selected_steps = steps

    steps_text = "\n".join(
        f"{step['order']}. {step['text']}"
        for step in selected_steps
    )

    # 지나치게 긴 텍스트 제한
    steps_text = steps_text[:2500]

    return {
        "recipe_uid": recipe["recipe_uid"],
        "title": recipe.get("title", ""),
        "description": recipe.get("description", ""),
        "ingredients": ", ".join(ingredient_names),
        "steps": steps_text,
    }

def format_number(value):
    """200.0 -> '200', 0.5 -> '0.5'"""
    if value is None:
        return None

    if float(value).is_integer():
        return str(int(value))

    return str(value)


def get_quantity(item):
    """
    cleaned 데이터에서 관계의 quantity 생성.

    200g     -> quantity='200'
    1~1.5개  -> quantity='1~1.5'
    약간     -> quantity='약간'
    """

    qualitative = item.get("qualitative_amount")

    if qualitative:
        return qualitative

    amount_min = item.get("amount_min")
    amount_max = item.get("amount_max")

    if amount_min is not None:

        if amount_max is None or amount_min == amount_max:
            return format_number(amount_min)

        return (
            f"{format_number(amount_min)}"
            f"~{format_number(amount_max)}"
        )

    return None

def make_graph_row(recipe):

    recipe_uid = recipe["recipe_uid"]

    dish = {
        "recipe_uid": recipe_uid,
        "title": recipe.get("title"),
        "views": recipe.get("views"),
        "servings": recipe.get("servings"),
        "cooking_time": recipe.get("cooking_time"),
        "difficulty": recipe.get("difficulty"),
        "source_url": recipe.get("source_url"),
        "cooking_method": recipe.get("cooking_method"),
    }

    ingredients = []

    for idx, item in enumerate(
        recipe.get("ingredients_clean", [])
    ):

        name = item.get("name_normalized")

        if not name:
            continue

        ingredients.append({
            "usage_key": (
                f"{recipe_uid}|food|{idx}|{name}"
            ),

            "name_normalized": name,

            "quantity": get_quantity(item),

            "unit": item.get("unit_normalized"),

            "preparation": item.get("preparation") or [],

            "role": "food",

            "raw": item.get("raw"),

            "amount_text": item.get("amount_text"),

            "source_group": item.get("group"),
        })


    offset = len(ingredients)

    for idx, item in enumerate(
        recipe.get("seasonings_clean", [])
    ):

        name = item.get("name_normalized")

        if not name:
            continue

        ingredients.append({
            "usage_key": (
                f"{recipe_uid}|seasoning|{offset + idx}|{name}"
            ),

            "name_normalized": name,

            "quantity": get_quantity(item),

            "unit": item.get("unit_normalized"),

            "preparation": item.get("preparation") or [],

            "role": "seasoning",

            "raw": item.get("raw"),

            "amount_text": item.get("amount_text"),

            "source_group": item.get("group"),
        })

    dish_group = recipe["dish_group"]
    dish_type = recipe["dish_type"]

    return {
        "dish_group": dish_group,

        "dish_type": dish_type,

        # 같은 이름의 DishType이 다른 대분류에 생기는 문제 방지
        "dish_type_key": f"{dish_group}::{dish_type}",

        "dish": dish,

        "ingredients": ingredients,
    }


def split_graph_rows(graph_rows):
    """원래 graph_rows를 노드 4종, 관계 3종 목록으로 나눕니다."""
    nodes = {label: {} for label in ("DishGroup", "DishType", "Dish", "Ingredient")}
    relations = {kind: {} for kind in ("HAS_TYPE", "HAS_DISH", "INGREDIENT")}
    for row in graph_rows:
        group, type_name, type_key = row["dish_group"], row["dish_type"], row["dish_type_key"]
        dish = dict(row["dish"])
        uid = dish["recipe_uid"]
        for value in (group, type_name, type_key, uid):
            if not isinstance(value, str) or not value.strip():
                raise ValueError("분류명 또는 식별자가 비어 있습니다.")
        if uid in nodes["Dish"]:
            raise ValueError(f"중복 recipe_uid: {uid}")
        if type_key != f"{group}::{type_name}":
            raise ValueError(f"대분류/중분류 키 불일치: {type_key}")
        nodes["DishGroup"][group] = {"name": group}
        nodes["DishType"][type_key] = {"key": type_key, "name": type_name}
        nodes["Dish"][uid] = dish
        relations["HAS_TYPE"][(group, type_key)] = {"source": group, "target": type_key}
        relations["HAS_DISH"][(type_key, uid)] = {"source": type_key, "target": uid}
        for ingredient in row["ingredients"]:
            name = ingredient["name_normalized"]
            usage_key = ingredient["usage_key"]
            if not isinstance(name, str) or not name.strip() or not usage_key:
                raise ValueError(f"재료 식별자 누락: {uid}")
            nodes["Ingredient"][name] = {"name_normalized": name}
            properties = {key: value for key, value in ingredient.items() if key != "name_normalized"}
            key = (uid, name, usage_key)
            if key in relations["INGREDIENT"]:
                raise ValueError(f"중복 재료 사용 관계: {usage_key}")
            # None도 보냅니다. 원본 SET과 동일하게 DB의 해당 속성이 제거됩니다.
            relations["INGREDIENT"][key] = {
                "source": uid, "target": name, "properties": properties,
            }
    if not nodes["Dish"]:
        raise ValueError("분류를 통과한 적재 대상이 없습니다.")
    node_rows = {label: list(values.values()) for label, values in nodes.items()}
    edge_rows = {kind: list(values.values()) for kind, values in relations.items()}
    expected = {name: len(rows) for name, rows in {**node_rows, **edge_rows}.items()}
    return {"nodes": node_rows, "relations": edge_rows, "expected": expected}


def run_batches(run_cypher, query, rows, batch_size=200):
    if type(batch_size) is not int or batch_size < 1:
        raise ValueError("batch_size는 양의 정수여야 합니다.")
    total = 0
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        result = run_cypher(query, rows=batch)
        processed = result[0]["processed"] if result else 0
        if processed != len(batch):
            raise RuntimeError(
                f"요청 {len(batch)}행 / 처리 {processed}행: 끝점 누락이나 중복 관계를 확인하세요. "
                "이미 처리한 배치는 자동 취소되지 않습니다."
            )
        total += processed
    return total


In [ ]:
if SOURCE_MODE == "reuse":
    classified_recipes = read_recipe_jsonl(CLASSIFIED_INPUT_PATH)
    print("기존 분류 파일:", CLASSIFIED_INPUT_PATH)
    print("읽은 레시피:", len(classified_recipes))
    print("LLM 분류 단계는 건너뜁니다.")
elif SOURCE_MODE == "classify":
    recipes = read_recipe_jsonl(RAW_INPUT_PATH, limit=MAX_RECIPES)
    print("원본 파일:", RAW_INPUT_PATH)
    print("분류할 레시피:", len(recipes))
else:
    raise ValueError("SOURCE_MODE는 reuse 또는 classify여야 합니다.")


In [ ]:
if SOURCE_MODE == "classify":
    from typing import Literal
    from pydantic import BaseModel, Field
    
    
    class DishClassification(BaseModel):
        recipe_uid: str = Field(
            description="입력으로 받은 레시피의 recipe_uid를 그대로 반환"
        )
    
        dish_group: Literal[
            "국",
            "찌개",
            "탕",
            "전골",
            "조림",
            "볶음",
            "구이",
            "찜",
            "튀김",
            "전/부침",
            "무침",
            "샐러드",
            "밥",
            "죽",
            "면/국수",
            "절임/장아찌",
            "소스/양념",
            "빵/베이킹",
            "디저트",
            "음료",
            "기타",
        ] = Field(
            description="완성 음식의 대표적인 조리 형태"
        )
    
        dish_type: str = Field(
            description=(
                "레시피가 실제로 의미하는 대표 음식명. "
                "광고 문구, 조리 팁, 사람 이름, 황금레시피 등의 표현은 제거한다."
            )
        )

In [ ]:
if SOURCE_MODE == "classify":
    from dotenv import load_dotenv
    from langchain_openai import ChatOpenAI

    if not ENV_PATH.is_file():
        raise FileNotFoundError(f".env 파일을 확인하세요: {ENV_PATH}")
    load_dotenv(ENV_PATH, override=True)
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError(".env에 OPENAI_API_KEY를 설정하세요.")

    MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")

    def make_model():
        return ChatOpenAI(model=MODEL_NAME)

    print("원본 기본값 또는 .env에서 선택한 모델:", MODEL_NAME)


In [ ]:
if SOURCE_MODE == "classify":
    from langchain_core.prompts import ChatPromptTemplate
    
    
    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """
    너는 한국 음식 레시피를 지식 그래프용으로 분류하는 전문가다.
    
    주어진 레시피를 분석해서 다음 두 가지를 결정한다.
    
    1. dish_group
    - 음식의 대표적인 조리 형태를 의미한다.
    - 반드시 허용된 DishGroup 중 하나를 선택한다.
    - 레시피 과정 중 잠깐 등장하는 조리법이 아니라,
      최종적으로 완성되는 음식의 형태를 기준으로 판단한다.
    
    예시:
    - 김치찌개 → 찌개
    - 된장찌개 → 찌개
    - 닭볶음탕 → 탕
    - 갈비탕 → 탕
    - 미역국 → 국
    - 감자조림 → 조림
    - 오징어볶음 → 볶음
    - 제육볶음 → 볶음
    - 갈비찜 → 찜
    - 생선구이 → 구이
    
    
    2. dish_type
    - 실제 음식 종류를 나타내는 대표 음식명이다.
    - 원본 title을 그대로 복사하지 않는다.
    - 음식 이름만 남긴다.
    - 같은 음식은 최대한 동일한 이름으로 통일한다.
    
    dish_type에서 제거해야 하는 표현:
    - 황금레시피
    - 만드는 법
    - 만들기
    - 맛있게 만드는 법
    - 초간단
    - 백종원
    - 엄마표
    - 집밥
    - 레시피
    - 비법
    - 추천
    - 최고의
    - 진짜진짜
    - 쉽고 간단한
    등 음식 자체의 이름이 아닌 표현
    
    예시:
    "돼지고기 김치찌개 맛내는 비법"
    → 김치찌개
    
    "오징어 볶음, 향과 맛이 일품! 백종원 오징어 볶음"
    → 오징어볶음
    
    "엄마의 레시피, 소고기 미역국 끓이는 법"
    → 소고기미역국
    
    "닭볶음탕 진짜진짜 황금레시피 알려 드려요"
    → 닭볶음탕
    
    
    중요한 규칙:
    
    - title 자체는 수정하거나 생성하지 않는다.
    - 입력에 없는 새로운 음식을 임의로 만들어내지 않는다.
    - 음식 이름이 title에 명확하게 존재하면 title을 우선한다.
    - title만으로 애매하면 description, 재료, 조리 과정을 참고한다.
    - 주요 재료가 음식 정체성에 중요한 경우 dish_type에 포함한다.
      예: 소고기미역국, 참치김치찌개, 오징어볶음
    - 단순한 홍보 문구나 조리 특징은 dish_type에서 제거한다.
    - 준비 과정에서 잠깐 볶는다고 해서 무조건 '볶음'으로 분류하지 않는다.
    - 최종 완성 음식의 형태를 기준으로 dish_group을 정한다.
    - 정말 분류하기 어려운 경우에만 dish_group을 '기타'로 한다.
    
    recipe_uid는 입력값을 한 글자도 변경하지 않고 그대로 반환한다.
    """
        ),
        (
            "human",
            """
    [recipe_uid]
    {recipe_uid}
    
    [title]
    {title}
    
    [description]
    {description}
    
    [주요 재료]
    {ingredients}
    
    [조리 과정]
    {steps}
    """
        )
    ])
    structured_llm = make_model().with_structured_output(DishClassification)
    classification_chain = prompt | structured_llm


In [ ]:
if SOURCE_MODE == "classify":
    batch_inputs = [make_llm_input(recipe) for recipe in recipes]
    print("분류 입력 개수:", len(batch_inputs))
    print("첫 입력:", batch_inputs[0])


In [ ]:
if SOURCE_MODE == "classify":
    if not ALLOW_LLM_CALLS:
        raise RuntimeError("LLM 호출 중단: 입력 개수와 모델을 확인하고 ALLOW_LLM_CALLS=True로 변경하세요.")

    from tqdm.auto import tqdm
    classification_results = []
    failed_results = []

    for start in tqdm(range(0, len(batch_inputs), LLM_BATCH_SIZE)):
        batch = batch_inputs[start:start + LLM_BATCH_SIZE]
        results = classification_chain.batch(
            batch,
            config={"max_concurrency": MAX_CONCURRENCY},
            return_exceptions=True,
        )
        if len(results) != len(batch):
            raise RuntimeError("분류 요청 수와 반환 수가 다릅니다.")
        for input_data, result in zip(batch, results):
            if isinstance(result, Exception):
                failed_results.append({"recipe_uid": input_data["recipe_uid"], "error": str(result)})
            elif not isinstance(result, DishClassification):
                failed_results.append({"recipe_uid": input_data["recipe_uid"], "error": "출력 형식 불일치"})
            elif result.recipe_uid != input_data["recipe_uid"]:
                failed_results.append({"recipe_uid": input_data["recipe_uid"], "error": "반환 recipe_uid 불일치"})
            elif not result.dish_type.strip():
                failed_results.append({"recipe_uid": input_data["recipe_uid"], "error": "빈 dish_type"})
            else:
                classification_results.append(result)

    print("형식/식별자 검사 통과:", len(classification_results))
    print("실패:", len(failed_results))
    print(failed_results[:5])


In [ ]:
if SOURCE_MODE == "classify":
    classification_map = {result.recipe_uid: result for result in classification_results}
    classified_recipes = []
    for recipe in recipes:
        result = classification_map.get(recipe["recipe_uid"])
        new_recipe = recipe.copy()
        new_recipe["dish_group"] = result.dish_group if result is not None else None
        new_recipe["dish_type"] = result.dish_type if result is not None else None

        steps = sorted(recipe.get("steps", []), key=lambda item: item.get("order", 9999))
        new_recipe["cooking_method"] = "\n".join(
            f"{step['order']}. {step['text'].strip()}"
            for step in steps if step.get("text")
        )
        classified_recipes.append(new_recipe)

for recipe in classified_recipes[:10]:
    print(recipe.get("recipe_uid"), "/", recipe.get("title"),
          "/", recipe.get("dish_group"), "/", recipe.get("dish_type"))

print("전체:", len(classified_recipes))
print("dish_group 누락:", sum(not row.get("dish_group") for row in classified_recipes))
print("dish_type 누락:", sum(not row.get("dish_type") for row in classified_recipes))


In [ ]:
if SOURCE_MODE == "classify" and ALLOW_FILE_WRITE:
    from datetime import datetime
    result_dir = WORK_DIR / "output" / ("classification_lesson_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    result_dir.mkdir(parents=True, exist_ok=False)
    saved_path = result_dir / "recipes_classified.jsonl"
    failure_path = result_dir / "classification_failed.jsonl"
    marker = result_dir / "INCOMPLETE"
    marker.touch(exist_ok=False)
    with saved_path.open("x", encoding="utf-8") as file:
        for recipe in classified_recipes:
            file.write(json.dumps(recipe, ensure_ascii=False, allow_nan=False) + "\n")
    with failure_path.open("x", encoding="utf-8") as file:
        for row in failed_results:
            file.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
    marker.unlink()
    print("새 결과:", saved_path)
    print("분류 실패 목록:", failure_path)
else:
    print("새 파일 저장 안 함: reuse 모드이거나 ALLOW_FILE_WRITE=False입니다.")


In [ ]:
invalid_recipes = [
    row for row in classified_recipes
    if not row.get("dish_group") or not row.get("dish_type")
]
valid_recipes = [
    row for row in classified_recipes
    if row.get("dish_group") and row.get("dish_type")
]
print("전체:", len(classified_recipes))
print("분류 누락으로 적재 제외:", len(invalid_recipes))
for row in invalid_recipes[:5]:
    print(row.get("recipe_uid"), row.get("title"))

graph_rows = [make_graph_row(recipe) for recipe in valid_recipes]
tables = split_graph_rows(graph_rows)
expected = tables["expected"]
for name, count in expected.items():
    print(f"{name:15s}: {count:,}")
print("예상 노드 합계:", sum(len(rows) for rows in tables["nodes"].values()))
print("예상 관계 합계:", sum(len(rows) for rows in tables["relations"].values()))
print("재료 사용 항목이 없는 Dish:", sum(not row["ingredients"] for row in graph_rows))
print("재료명이 없어 건너뛴 항목:", sum(
    not item.get("name_normalized")
    for recipe in valid_recipes
    for field in ("ingredients_clean", "seasonings_clean")
    for item in recipe.get(field, [])
))


In [ ]:
if not ALLOW_DB_WRITE:
    raise RuntimeError("DB 진행 중단: 대상 DB를 확인하고 ALLOW_DB_WRITE=True로 변경하세요.")

from dotenv import load_dotenv
from neo4j import GraphDatabase

if not ENV_PATH.is_file():
    raise FileNotFoundError(f".env 경로를 확인하세요: {ENV_PATH}")
load_dotenv(ENV_PATH, override=True)
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
if not all((NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)):
    raise ValueError("Neo4j 접속 정보가 비어 있습니다.")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
if globals().get("driver") is not None:
    driver.close()

try:
    driver.verify_connectivity()
except Exception:
    driver.close()
    raise

def run_cypher(query, **params):
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, **params)
        records = [record.data() for record in result]
        result.consume()
        return records

def require_db_write():
    if not ALLOW_DB_WRITE:
        raise RuntimeError("ALLOW_DB_WRITE=True일 때만 DB에 씁니다.")

print("연결 주소:", NEO4J_URI)
print("데이터베이스:", NEO4J_DATABASE)
print("DB 전체 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])
print("DB 전체 관계:", run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"])

old_model_count = run_cypher("""
MATCH (n)
WHERE n:Recipe OR n:Component OR (n:Dish AND n.name IS NOT NULL)
RETURN count(n) AS n
""")[0]["n"]
if old_model_count:
    driver.close()
    raise RuntimeError("이전 Recipe/Component/Dish(name) 모델이 보입니다. 다른 전용 DB를 선택하세요.")


In [ ]:
require_db_write()
constraint_specs = [
    ("dish_uid_unique", "Dish", "recipe_uid"),
    ("dish_group_unique", "DishGroup", "name"),
    ("dish_type_unique", "DishType", "key"),
    ("ingredient_unique", "Ingredient", "name_normalized"),
]
for name, label, key in constraint_specs:
    run_cypher(f"CREATE CONSTRAINT {name} IF NOT EXISTS FOR (n:{label}) REQUIRE n.{key} IS UNIQUE")

constraint_rows = run_cypher("""
SHOW CONSTRAINTS
YIELD name, type, entityType, labelsOrTypes, properties
RETURN name, type, entityType, labelsOrTypes, properties
ORDER BY name
""")
for row in constraint_rows:
    print(row)

missing = [
    f"{label}.{key}" for _, label, key in constraint_specs
    if not any(
        row["entityType"] == "NODE"
        and row["labelsOrTypes"] == [label]
        and row["properties"] == [key]
        and row["type"] in ("UNIQUENESS", "NODE_PROPERTY_UNIQUENESS", "NODE_KEY")
        for row in constraint_rows
    )
]
if missing:
    raise RuntimeError(f"필요한 제약 미확인: {missing}")


In [ ]:
require_db_write()
processed = run_batches(run_cypher, """
UNWIND $rows AS row
MERGE (g:DishGroup {name: row.name})
RETURN count(*) AS processed
""", tables["nodes"]["DishGroup"], DB_BATCH_SIZE)
print("DishGroup 처리:", processed)


In [ ]:
require_db_write()
processed = run_batches(run_cypher, """
UNWIND $rows AS row
MERGE (t:DishType {key: row.key})
SET t.name = row.name
RETURN count(*) AS processed
""", tables["nodes"]["DishType"], DB_BATCH_SIZE)
print("DishType 처리:", processed)


In [ ]:
require_db_write()
processed = run_batches(run_cypher, """
UNWIND $rows AS row
MERGE (d:Dish {recipe_uid: row.recipe_uid})
SET d += row
RETURN count(*) AS processed
""", tables["nodes"]["Dish"], DB_BATCH_SIZE)
print("Dish 처리:", processed)


In [ ]:
require_db_write()
processed = run_batches(run_cypher, """
UNWIND $rows AS row
MERGE (i:Ingredient {name_normalized: row.name_normalized})
RETURN count(*) AS processed
""", tables["nodes"]["Ingredient"], DB_BATCH_SIZE)
print("Ingredient 처리:", processed)


In [ ]:
require_db_write()
processed = run_batches(run_cypher, """
UNWIND $rows AS row
MATCH (g:DishGroup {name: row.source})
MATCH (t:DishType {key: row.target})
MERGE (g)-[:HAS_TYPE]->(t)
RETURN count(*) AS processed
""", tables["relations"]["HAS_TYPE"], DB_BATCH_SIZE)
print("HAS_TYPE 처리:", processed)


In [ ]:
require_db_write()
processed = run_batches(run_cypher, """
UNWIND $rows AS row
MATCH (t:DishType {key: row.source})
MATCH (d:Dish {recipe_uid: row.target})
MERGE (t)-[:HAS_DISH]->(d)
RETURN count(*) AS processed
""", tables["relations"]["HAS_DISH"], DB_BATCH_SIZE)
print("HAS_DISH 처리:", processed)


In [ ]:
require_db_write()
processed = run_batches(run_cypher, """
UNWIND $rows AS row
MATCH (d:Dish {recipe_uid: row.source})
MATCH (i:Ingredient {name_normalized: row.target})
MERGE (d)-[r:INGREDIENT {usage_key: row.properties.usage_key}]->(i)
SET r += row.properties
RETURN count(*) AS processed
""", tables["relations"]["INGREDIENT"], DB_BATCH_SIZE)
print("INGREDIENT 처리:", processed)


In [ ]:
node_keys = {"DishGroup": "name", "DishType": "key", "Dish": "recipe_uid", "Ingredient": "name_normalized"}
edge_specs = {
    "HAS_TYPE": ("DishGroup", "name", "DishType", "key"),
    "HAS_DISH": ("DishType", "key", "Dish", "recipe_uid"),
    "INGREDIENT": ("Dish", "recipe_uid", "Ingredient", "name_normalized"),
}
checks = []
for label, key in node_keys.items():
    query = f"UNWIND $rows AS row MATCH (n:{label} {{{key}: row.{key}}}) RETURN count(n) AS n"
    rows = tables["nodes"][label]
    actual = sum(run_cypher(query, rows=rows[start:start+DB_BATCH_SIZE])[0]["n"]
                 for start in range(0, len(rows), DB_BATCH_SIZE))
    checks.append((label, expected[label], actual))

for kind, (src_label, src_key, dst_label, dst_key) in edge_specs.items():
    rel_match = "[r:INGREDIENT {usage_key: row.properties.usage_key}]" if kind == "INGREDIENT" else f"[r:{kind}]"
    query = (
        "UNWIND $rows AS row "
        f"MATCH (a:{src_label} {{{src_key}: row.source}})-{rel_match}->"
        f"(b:{dst_label} {{{dst_key}: row.target}}) RETURN count(r) AS n"
    )
    rows = tables["relations"][kind]
    actual = sum(run_cypher(query, rows=rows[start:start+DB_BATCH_SIZE])[0]["n"]
                 for start in range(0, len(rows), DB_BATCH_SIZE))
    checks.append((kind, expected[kind], actual))

for name, wanted, actual in checks:
    print(f"{name:15s} 예상={wanted:,} / 실제={actual:,} / 일치={wanted == actual}")
if any(wanted != actual for _, wanted, actual in checks):
    raise RuntimeError("누락 또는 중복이 있습니다. 이미 적재된 배치는 자동 취소되지 않습니다.")


In [ ]:
validation_rows = [
    {"recipe_uid": row["dish"]["recipe_uid"], "type_key": row["dish_type_key"],
     "ingredient_count": len(row["ingredients"])}
    for row in graph_rows
]
issues = []
query = """
UNWIND $rows AS row
MATCH (d:Dish {recipe_uid: row.recipe_uid})
OPTIONAL MATCH (t)-[:HAS_DISH]->(d)
WITH row, d, count(t) AS types, collect(t.key) AS type_keys
OPTIONAL MATCH (d)-[r:INGREDIENT]->()
WITH row, types, type_keys, count(r) AS uses
WHERE types <> 1 OR NOT row.type_key IN type_keys OR uses <> row.ingredient_count
RETURN row.recipe_uid AS recipe_uid, types, type_keys, uses
"""
for start in range(0, len(validation_rows), DB_BATCH_SIZE):
    issues.extend(run_cypher(query, rows=validation_rows[start:start+DB_BATCH_SIZE]))
print("분류/재료 연결 점검 대상:", len(issues))
print(issues[:10])
if issues:
    raise RuntimeError("예전 연결 또는 현재 적재 내용을 확인하세요. 자동 삭제하지 않았습니다.")


In [ ]:
for row in run_cypher("""
MATCH (g:DishGroup)-[:HAS_TYPE]->(t:DishType)-[:HAS_DISH]->(d:Dish {recipe_uid: $uid})
OPTIONAL MATCH (d)-[r:INGREDIENT]->(i:Ingredient)
RETURN g.name AS dish_group, t.name AS dish_type, d.title AS title,
       i.name_normalized AS ingredient, r.role AS role, r.quantity AS quantity, r.unit AS unit
ORDER BY role, ingredient
""", uid=graph_rows[0]["dish"]["recipe_uid"]):
    print(row)


In [ ]:
driver.close()
print('Neo4j 연결 종료')
